# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark RDD - SOLUTION
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful

In [1]:
from pyspark import SparkContext, SparkConf
import numpy as np
import operator

In [2]:
conf=SparkConf().setAppName("Lab4-rdd").setMaster("local[*]")
sc = SparkContext(conf=conf)

Using PySpark and RDD's on the https://coding.csel.io machines is slow -- most of the code is executed in Python and this is much less efficient than the java-based code using the PySpark dataframes. Be patient and trying using `.cache()` to cache the output of joins. You may want to start with a reduced set of data before running the full task. You can use the `sample()` method to extract just a sample of the data or use 

These two RDD's are called "rawCitations" and "rawPatents" because you probably want to process them futher (e.g. convert them to integer types, etc). 

The `textFile` function returns data in strings. This should work fine for this lab.

Other methods you use might return data in type `Byte`. If you haven't used Python `Byte` types before, google it. You can convert a value of `x` type byte into e.g. a UTF8 string using `x.decode('uft-8')`. Alternatively, you can use the `open` method of the gzip library to read in all the lines as UTF-8 strings like this:
```
import gzip
with gzip.open('cite75_99.txt.gz', 'rt',encoding='utf-8') as f:
    rddCitations = sc.parallelize( f.readlines() )
```
This is less efficient than using `textFile` because `textFile` would use the underlying HDFS or other file system to read the file across all the worker nodes while the using `gzip.open()...readlines()` will read all the data in the frontend and then distribute it to all the worker nodes.

In [3]:
rddCitations = sc.textFile("cite75_99.txt.gz")
rddPatents = sc.textFile("apat63_99.txt.gz")

The data looks like the following.

In [4]:
rddCitations.take(5)

['"CITING","CITED"',
 '3858241,956203',
 '3858241,1324234',
 '3858241,3398406',
 '3858241,3557384']

In [5]:
rddPatents.take(5)

['"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,',
 '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,',
 '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,']

In other words, they are a single string with multiple CSV's. You will need to convert these to (K,V) pairs, probably convert the keys to `int` and so on. You'll need to `filter` out the header string as well since there's no easy way to extract all the lines except the first.

In [6]:
# sample for testing
# rddPatents = rddPatents.sample(False, 0.05, seed=42)
# rddCitations = rddCitations.sample(False, 0.05, seed=42)

Notes: We first have to extract the headers, because they are included in the RDD. We can use the `first()` method to do this

In [7]:
patentsHeader = rddPatents.first()
citationsHeader = rddCitations.first()

patentsHeader, citationsHeader

('"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '"CITING","CITED"')

Notes: we use the headers from above and filter them out of the RDD, so that the remaining lines consist purely of actual patent or citation data.

In [8]:
# filter out headers, split csv data along commas
patents = rddPatents.filter(lambda line : line != patentsHeader).map(lambda line : line.strip().split(","))
citations = rddCitations.filter(lambda line : line != citationsHeader).map(lambda line : line.strip().split(","))

Notes: the rest of the process is donne in cell below. I tried to comment where I felt was necessary. But an overview is given here as well. We first have to key the patents table by patent number (with its state as the value), and the citations table by citing patent. This lets us do a key/value join of both RDDs on matching patent number, obtaining the citing patent's state. We have to do a similar operation to get the state for the cited patent by keying the RDD by the citing patent, and joining the initial patent RDD. This gives us both states for citing and cited patent. We can filter out null or empty values for state, as well as filter by matching states. Then we reduce over citing patent by adding the total same state counts. Like the dataframe solution, we cana now just join this with the original patents RDD and add a value. This is done by keying both by patent number and joining them. We order them by same state count and print the results.

In [12]:
# want to use citations and patents to pair citing, cited with their states
patent_idx = 0
state_idx = 5

patentState = patents.map(lambda fields : (fields[patent_idx], fields[state_idx])).cache()
citationPairs = citations.map(lambda fields : (fields[0], fields[1])).cache() # citing, cited

# join on citing state
citingAndState = citationPairs.join(patentState).cache()
# should be (citing_patent, (cited_patent, citing_state)) or similar

# make cited the new key to join for its state later
citedKey = citingAndState.map(lambda kv : (kv[1][0], (kv[0], kv[1][1])))
# should be (cited_patent, (citing_patent, citing_state)) or similar

bothStates = citedKey.join(patentState)
# should get (cited_patent, ((citing_patent, citing_state), cited_state)
# filter where states match and state is not null or empty
"""
so: kv[1][0][1] = citing_state
kv[1][1] = cited_state
"""
sameStates = bothStates.filter(lambda kv : kv[1][0][1] == kv[1][1] and kv[1][0][1] not in (None, '', '""'))

# group by citing, with count
"""
kv[1][0][0] = citing_patent
"""
sameStateCounts = sameStates.map(lambda kv : (kv[1][0][0], 1)).reduceByKey(lambda a,b : a+b).cache()

# prepare to join with same state counts
patentsKey = patents.map(lambda fields : (fields[patent_idx], fields))
joined = patentsKey.leftOuterJoin(sameStateCounts)
# should look like: (patent, (fields, same state count))

# now order by 
# top15 = joined.sortBy(lambda kv : (kv[1][1] if kv[1][1] is not None else -1), ascending=False).take(15)
top15 = joined.takeOrdered(15, key=lambda kv : -(kv[1][1] if kv[1][1] is not None else -1))

# top15

In [13]:
# better formatting for print
for patent_id, (fields, count) in top15:
    print(fields, count)

['5959466', '1999', '14515', '1997', '"US"', '"CA"', '5310', '2', '', '326', '4', '46', '159', '0', '1', '', '0.6186', '', '4.8868', '0.0455', '0.044', '', ''] 125
['5983822', '1999', '14564', '1998', '"US"', '"TX"', '569900', '2', '', '114', '5', '55', '200', '0', '0.995', '', '0.7201', '', '12.45', '0', '0', '', ''] 103
['6008204', '1999', '14606', '1998', '"US"', '"CA"', '749584', '2', '', '514', '3', '31', '121', '0', '1', '', '0.7415', '', '5', '0.0085', '0.0083', '', ''] 100
['5952345', '1999', '14501', '1997', '"US"', '"CA"', '749584', '2', '', '514', '3', '31', '118', '0', '1', '', '0.7442', '', '5.1102', '0', '0', '', ''] 98
['5998655', '1999', '14585', '1998', '"US"', '"CA"', '', '1', '', '560', '1', '14', '114', '0', '1', '', '0.7387', '', '5.1667', '', '', '', ''] 96
['5958954', '1999', '14515', '1997', '"US"', '"CA"', '749584', '2', '', '514', '3', '31', '116', '0', '1', '', '0.7397', '', '5.181', '0', '0', '', ''] 96
['5936426', '1999', '14466', '1997', '"US"', '"CA"', '5